In [2]:
import pandas as pd
import numpy as np

customers = pd.read_csv("data/cleaned/customers.csv")
transactions = pd.read_csv("data/cleaned/transactions.csv")

In [3]:
customers.head()
transactions.head()

customers.info()
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   customer_id     10000 non-null  int64
 1   age             10000 non-null  int64
 2   gender          10000 non-null  str  
 3   income_bracket  10000 non-null  str  
 4   income_range    10000 non-null  str  
 5   occupation      10000 non-null  str  
 6   city_tier       10000 non-null  str  
 7   join_date       10000 non-null  str  
 8   churn_flag      10000 non-null  str  
 9   age_group       10000 non-null  str  
dtypes: int64(2), str(8)
memory usage: 781.4 KB
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    100000 non-null  int64  
 1   customer_id       100000 non-null  int64  
 2   transaction_date  100000 non-null  str    
 

In [4]:
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])

In [5]:
transactions['year_month'] = transactions['transaction_date'].dt.to_period('M')

In [6]:
transactions.head(10)

,transaction_id,customer_id,transaction_date,amount,category,transaction_type,channel,is_salary_credit,year_month
0,1,6685,2022-01-27,5930.64,Fuel,Debit,Card,No,2022-01
1,2,7553,2022-11-08,3400.38,Education,Debit,Card,No,2022-11
2,3,2715,2023-05-02,1376.27,Education,Debit,Card,No,2023-05
3,4,6756,2022-10-27,2543.37,Travel,Debit,ATM,No,2022-10
4,5,9635,2024-02-26,9210.60,Transfer,Credit,UPI,No,2024-02
5,6,2640,2023-09-14,1010.52,Transfer,Debit,UPI,No,2023-09
6,7,6685,2023-08-14,29.46,Travel,Credit,Branch,Yes,2023-08
7,8,4158,2022-02-02,2061.58,Shopping,Debit,ATM,No,2022-02
8,9,1919,2022-06-09,6896.37,Fuel,Debit,Branch,No,2022-06
9,10,5922,2022-01-27,150.22,Shopping,Debit,Card,No,2022-01


In [7]:
customers.head(10)

,customer_id,age,gender,income_bracket,income_range,occupation,city_tier,join_date,churn_flag,age_group
0,1,56,Male,Medium,4-8 Lakh,Salaried,Tier 3,8/26/2023,No,Pre-Retirement
1,2,46,Male,High,8+ Lakh,Salaried,Tier 1,4/22/2020,Yes,Family Stage
2,3,32,Female,High,8+ Lakh,Self-Employed,Tier 1,7/17/2023,No,Mid Career
3,4,60,Female,Medium,4-8 Lakh,Self-Employed,Tier 2,11/15/2020,No,Pre-Retirement
4,5,25,Female,High,8+ Lakh,Business,Tier 3,5/23/2025,No,Early Career
5,6,38,Female,Medium,4-8 Lakh,Business,Tier 3,6/23/2020,No,Mid Career
6,7,56,Male,High,8+ Lakh,Salaried,Tier 1,1/19/2025,No,Pre-Retirement
7,8,36,Female,Low,2-4 Lakh,Business,Tier 2,11/8/2022,No,Mid Career
8,9,40,Male,Low,2-4 Lakh,Business,Tier 2,5/11/2023,No,Family Stage
9,10,28,Male,Medium,4-8 Lakh,Self-Employed,Tier 3,5/12/2025,No,Early Career


In [8]:
transactions['debit_amount'] = np.where(
    transactions['transaction_type'] == 'Debit',
    transactions['amount'], 0
)

transactions['credit_amount'] = np.where(
    transactions['transaction_type'] == 'Credit',
    transactions['amount'], 0
)

In [9]:
transactions['salary_amount'] = np.where(
    transactions['is_salary_credit'] == 'Yes',
    transactions['amount'], 0
)

In [10]:
total_spend = transactions.groupby('customer_id')['debit_amount'].sum().reset_index()
total_salary = transactions.groupby('customer_id')['salary_amount'].sum().reset_index()
txn_count = transactions.groupby('customer_id')['transaction_id'].count().reset_index()

In [11]:
customer_summary = customers.merge(total_spend, on='customer_id', how='left') \
                            .merge(total_salary, on='customer_id', how='left') \
                            .merge(txn_count, on='customer_id', how='left')

In [12]:
customer_summary['savings_ratio'] = (
    customer_summary['salary_amount'] / customer_summary['debit_amount']
)

In [13]:
customer_summary['savings_ratio'] = customer_summary['savings_ratio'].replace([np.inf, -np.inf], 0)
customer_summary['savings_ratio'] = customer_summary['savings_ratio'].fillna(0)

In [14]:
customer_summary['spend_segment'] = pd.cut(
    customer_summary['debit_amount'],
    bins=[0, 20000, 50000, 1000000],
    labels=['Low', 'Medium', 'High']
)

In [15]:
customer_summary.to_csv("data/cleaned/customer_summary.csv", index=False)

In [16]:
customer_summary.head(10)

,customer_id,age,gender,income_bracket,income_range,occupation,city_tier,join_date,churn_flag,age_group,debit_amount,salary_amount,transaction_id,savings_ratio,spend_segment
0,1,56,Male,Medium,4-8 Lakh,Salaried,Tier 3,8/26/2023,No,Pre-Retirement,10785.43,3374.24,13.0,0.312852,Low
1,2,46,Male,High,8+ Lakh,Salaried,Tier 1,4/22/2020,Yes,Family Stage,7852.89,302.97,10.0,0.038581,Low
2,3,32,Female,High,8+ Lakh,Self-Employed,Tier 1,7/17/2023,No,Mid Career,3258.14,0.00,4.0,0.000000,Low
3,4,60,Female,Medium,4-8 Lakh,Self-Employed,Tier 2,11/15/2020,No,Pre-Retirement,18073.13,0.00,11.0,0.000000,Low
4,5,25,Female,High,8+ Lakh,Business,Tier 3,5/23/2025,No,Early Career,11831.79,0.00,7.0,0.000000,Low
5,6,38,Female,Medium,4-8 Lakh,Business,Tier 3,6/23/2020,No,Mid Career,9560.54,822.44,8.0,0.086024,Low
6,7,56,Male,High,8+ Lakh,Salaried,Tier 1,1/19/2025,No,Pre-Retirement,37224.94,0.00,10.0,0.000000,Medium
7,8,36,Female,Low,2-4 Lakh,Business,Tier 2,11/8/2022,No,Mid Career,20214.62,0.00,12.0,0.000000,Medium
8,9,40,Male,Low,2-4 Lakh,Business,Tier 2,5/11/2023,No,Family Stage,3339.78,0.00,5.0,0.000000,Low
9,10,28,Male,Medium,4-8 Lakh,Self-Employed,Tier 3,5/12/2025,No,Early Career,14186.38,0.00,8.0,0.000000,Low


In [17]:
customer_summary.rename(columns={'transaction_id': 'transaction_count'}, inplace=True)

In [18]:
customer_summary.isnull().sum()

customer_id          0
age                  0
gender               0
income_bracket       0
income_range         0
occupation           0
city_tier            0
join_date            0
churn_flag           0
age_group            0
debit_amount         1
salary_amount        1
transaction_count    1
savings_ratio        0
spend_segment        1
dtype: int64

In [19]:
customer_summary.describe()

,customer_id,age,debit_amount,salary_amount,transaction_count,savings_ratio
count,10000.00000,10000.000000,9999.000000,9999.000000,9999.000000,10000.000000
mean,5000.50000,41.054000,16114.621966,1176.891852,10.001000,0.110450
std,2886.89568,13.484104,8151.138632,2159.219238,3.113702,0.426759
min,1.00000,18.000000,201.280000,0.000000,1.000000,0.000000
25%,2500.75000,30.000000,10174.670000,0.000000,8.000000,0.000000
50%,5000.50000,41.000000,15090.190000,0.000000,10.000000,0.000000
75%,7500.25000,53.000000,20806.975000,1533.120000,12.000000,0.103535
max,10000.00000,64.000000,61290.760000,22894.440000,24.000000,24.464924


In [20]:
customer_summary['savings_ratio'].describe()

count    10000.000000
mean         0.110450
std          0.426759
min          0.000000
25%          0.000000
50%          0.000000
75%          0.103535
max         24.464924
Name: savings_ratio, dtype: float64

In [21]:
customer_summary.groupby('churn_flag')[[
    'debit_amount',
    'salary_amount',
    'transaction_count',
    'savings_ratio'
]].mean()

,debit_amount,salary_amount,transaction_count,savings_ratio
churn_flag,,,,
No,16159.965168,1193.314461,10.027671,0.112137
Yes,15926.260696,1108.670304,9.890206,0.103447


In [22]:
customer_summary.to_csv("data/cleaned/customer_summary.csv", index=False)

In [24]:
transactions['debit_amount'] = np.where(
    transactions['transaction_type'] == 'Debit',
    transactions['amount'], 0
)

transactions['credit_amount'] = np.where(
    transactions['transaction_type'] == 'Credit',
    transactions['amount'], 0
)

transactions['salary_amount'] = np.where(
    transactions['is_salary_credit'] == 'Yes',
    transactions['amount'], 0
)

In [25]:
transactions.to_csv("data/cleaned/transactions.csv", index=False)